# 04. MPS crime × borough income

Prior-12-month MPS crime regressions test whether the crime–price association varies with borough income. The interaction is motivated by the possibility that crime may be concentrated in higher-value areas and by the idea that income proxies for neighbourhood economic context. Income is borough-level gross annual pay, not household income. Each model includes crime and income main effects, their interaction, property controls, all five location controls, and remaining economic controls. The interaction is associational and does not resolve reverse causality. M2–M7 follow notebook 03's model sequence.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from linearmodels.iv import AbsorbingLS
try:
 from google.colab import drive; drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/HE3027 Project')
except ImportError: ROOT=Path.cwd()
DATA=ROOT/'data'; SRC=DATA/'crime_housing_merged.csv.gz'
if not SRC.is_file(): raise FileNotFoundError('Run crime_housing_project_analysis.ipynb first.')
OUT=DATA/'04_crime_mps_interaction_outputs'; OUT.mkdir(exist_ok=True)
d=pd.read_csv(SRC,low_memory=False)
d['lnprice']=np.log(d.price.where(d.price.gt(0))); d['newbuild']=d.age.eq('Y').astype(float); d['grossannualpay_k']=d.grossannualpay/1000; d['Nearest_Park_dist_km']=d['Nearest_Park_dist']/1000
for v in ['tube_distnear','bus_distnear','thamesriv_dist']: d[v+'_km']=d[v]/1000
crime_types=['mps_all','mps_violence','mps_property']; loc=['tube_distnear_km','bus_distnear_km','thamesriv_dist_km','Nearest_Park_dist_km','dist_to_cbd']; prop=['detached_dum','semi_d_dum','terrace_dum','freehold','newbuild']; econ=['hoursworked','unemployment']
for c in crime_types: d['crime_'+c]=np.log1p(d[c+'_prior12'])
valid=d.mps_exposures_available & d.mps_geography_ok & d.analysis_geography_ok
need=['lnprice','lsoacode','msoa11','oslaua','year','yearquarter','grossannualpay_k']+loc+prop+econ+['crime_'+c for c in crime_types]
s=d.loc[valid,need].dropna().copy(); s['year_fe']=s.year.astype(str); s['yearquarter_fe']=s.yearquarter.astype(str)
s['income_c']=s.grossannualpay_k-s.grossannualpay_k.mean()
for c in crime_types:
 s['crime_'+c+'_c']=s['crime_'+c]-s['crime_'+c].mean(); s['int_'+c+'_income']=s['crime_'+c+'_c']*s.income_c
specs=[('M2',[],prop),('M3',[],prop+loc+econ),('M4',['oslaua','year_fe'],prop+loc+econ),('M5',['msoa11','year_fe'],prop+loc+econ),('M6',['msoa11','yearquarter_fe'],prop+loc+econ),('M7',['lsoacode','yearquarter_fe'],prop+loc+econ)]
rows=[]; coefs=[]
for c in crime_types:
 for model,fe,controls in specs:
  terms=['crime_'+c+'_c','income_c','int_'+c+'_income']; names=terms+controls; X=s[names].astype(float); scale=X.std().replace(0,1).to_numpy(); Xs=X/scale
  absorb=pd.DataFrame({k:pd.Categorical(s[k]) for k in fe},index=s.index) if fe else None
  fit=AbsorbingLS(s.lnprice,Xs,absorb=absorb,drop_absorbed=True).fit(cov_type='clustered',clusters=s.lsoacode,debiased=True)
  G=s.lsoacode.nunique(); crit=stats.t.ppf(.975,G-1); names_kept=list(fit.params.index)
  for v in names_kept:
   j=names.index(v); b=fit.params[v]/scale[j]; se=fit.std_errors[v]/scale[j]; coefs.append(dict(model=model,crime=c,variable=v,beta=b,se=se,ci_low=b-crit*se,ci_high=b+crit*se,p=2*stats.t.sf(abs(b/se),G-1),N=len(s),clusters=G))
  v='int_'+c+'_income'
  if v not in names_kept: raise ValueError(f'Interaction {v} is absorbed in {model}.')
  j=names.index(v); b=fit.params[v]/scale[j]; se=fit.std_errors[v]/scale[j]
  rows.append(dict(model=model,crime=c,data_source='MPS',exposure='prior12',method='Crime × borough income',N=len(s),clusters=G,beta=b,se=se,interaction_beta=b,interaction_se=se,ci_low=b-crit*se,ci_high=b+crit*se,p=2*stats.t.sf(abs(b/se),G-1),r2=fit.rsquared,fixed_effects=';'.join(fe) or 'none',absorbed_or_collinear='; '.join(v for v in names if v not in names_kept)))
results=pd.DataFrame(rows); coefficients=pd.DataFrame(coefs)
OUT.mkdir(parents=True,exist_ok=True)
results.to_csv(OUT/'interaction_regressions.csv',index=False); coefficients.to_csv(OUT/'interaction_all_coefficients.csv',index=False)
display(results); display(coefficients)


,model,crime,data_source,exposure,method,N,clusters,beta,se,interaction_beta,interaction_se,ci_low,ci_high,p,r2,fixed_effects,absorbed_or_collinear
0,M2,mps_all,MPS,prior12,Crime × borough income,338347,4647,0.345756,0.049085,0.345756,0.049085,0.249527,0.441986,2.140458e-12,0.575853,none,
1,M3,mps_all,MPS,prior12,Crime × borough income,338347,4647,0.006421,0.002471,0.006421,0.002471,0.001577,0.011266,9.389231e-03,0.998694,none,
2,M4,mps_all,MPS,prior12,Crime × borough income,338347,4647,0.007110,0.001955,0.007110,0.001955,0.003277,0.010943,2.796097e-04,0.595075,oslaua;year_fe,
3,M5,mps_all,MPS,prior12,Crime × borough income,338347,4647,0.001936,0.001240,0.001936,0.001240,-0.000495,0.004368,1.185785e-01,0.711981,msoa11;year_fe,
4,M6,mps_all,MPS,prior12,Crime × borough income,338347,4647,0.001958,0.001234,0.001958,0.001234,-0.000461,0.004377,1.125968e-01,0.715276,msoa11;yearquarter_fe,
5,M7,mps_all,MPS,prior12,Crime × borough income,338347,4647,-0.000240,0.001397,-0.000240,0.001397,-0.002980,0.002499,8.633964e-01,0.756130,lsoacode;yearquarter_fe,
6,M2,mps_violence,MPS,prior12,Crime × borough income,338347,4647,0.279106,0.040392,0.279106,0.040392,0.199919,0.358294,5.503232e-12,0.568884,none,
7,M3,mps_violence,MPS,prior12,Crime × borough income,338347,4647,0.002767,0.002323,0.002767,0.002323,-0.001788,0.007322,2.337131e-01,0.998749,none,
8,M4,mps_violence,MPS,prior12,Crime × borough income,338347,4647,0.001716,0.001809,0.001716,0.001809,-0.001830,0.005261,3.428771e-01,0.611636,oslaua;year_fe,
9,M5,mps_violence,MPS,prior12,Crime × borough income,338347,4647,-0.001959,0.001175,-0.001959,0.001175,-0.004263,0.000346,9.569417e-02,0.713229,msoa11;year_fe,


,model,crime,variable,beta,se,ci_low,ci_high,p,N,clusters
0,M2,mps_all,crime_mps_all_c,1.500003,0.182776,1.141674,1.858331,2.914515e-16,338347,4647
1,M2,mps_all,income_c,0.336191,0.037025,0.263604,0.408778,1.572000e-19,338347,4647
2,M2,mps_all,int_mps_all_income,0.345756,0.049085,0.249527,0.441986,2.140458e-12,338347,4647
3,M2,mps_all,detached_dum,11.751982,0.148592,11.460671,12.043293,0.000000e+00,338347,4647
4,M2,mps_all,semi_d_dum,11.359981,0.134092,11.097096,11.622865,0.000000e+00,338347,4647
...,...,...,...,...,...,...,...,...,...,...
244,M7,mps_property,thamesriv_dist_km,-0.005699,0.015212,-0.035523,0.024125,7.079568e-01,338347,4647
245,M7,mps_property,Nearest_Park_dist_km,-0.027101,0.017348,-0.061112,0.006910,1.183166e-01,338347,4647
246,M7,mps_property,dist_to_cbd,-0.015572,0.017051,-0.049001,0.017857,3.611612e-01,338347,4647
247,M7,mps_property,hoursworked,-0.007546,0.002497,-0.012442,-0.002650,2.528189e-03,338347,4647


The interaction coefficient gives the change in the crime–price association for a GBP 1,000 increase in centered borough gross annual pay. A positive estimate means the crime association is more positive or less negative in higher-pay borough-years. Borough pay's main effect is retained when identified; separate borough and year effects do not automatically absorb a borough-year variable. The fixed-effects models identify the interaction from within-area and time variation. Standard errors are clustered by LSOA. This heterogeneity analysis does not establish the direction of causality.


## Figures and descriptive checks

The plots show the common sample, interaction estimates, and the implied crime slopes across observed borough-pay values. Slope bands use the full clustered covariance between the crime main effect and interaction.


In [2]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
s.groupby('year').size().plot.bar(ax=axes[0],color='#3465a4')
axes[0].set_title('Common sample by sale year');axes[0].set_ylabel('Sales');axes[0].set_xlabel('Year')
for c in crime_types:
    b=s[['grossannualpay_k','crime_'+c+'_c']].copy()
    b['pay_group']=pd.qcut(b.grossannualpay_k,4,duplicates='drop')
    means=b.groupby('pay_group',observed=True)['crime_'+c+'_c'].mean()
    axes[1].plot(range(len(means)),means.to_numpy(),marker='o',label=c.replace('mps_',''))
axes[1].set_xticks(range(len(means)),[f'Q{i+1}' for i in range(len(means))])
axes[1].set_title('Mean centered log crime by borough-pay quartile')
axes[1].set_xlabel('Borough annual pay');axes[1].set_ylabel('Mean centered log(1 + prior12 crime)');axes[1].legend()
fig.tight_layout();fig.savefig(OUT/'sample_and_income_patterns.png',dpi=170,bbox_inches='tight');plt.show()

fig,axes=plt.subplots(1,3,figsize=(17,4.8),sharey=True)
for ax,c in zip(axes,crime_types):
    b=results.loc[results.crime.eq(c)].set_index('model').reindex([x[0] for x in specs])
    ax.errorbar(range(len(b)),b.beta,yerr=np.vstack([b.beta-b.ci_low,b.ci_high-b.beta]),fmt='o',capsize=4)
    ax.axhline(0,color='grey',linestyle='--');ax.set_xticks(range(len(b)),b.index)
    ax.set_title(c.replace('mps_','').title());ax.set_xlabel('Model')
axes[0].set_ylabel('Crime × borough pay coefficient (95% clustered CI)')
fig.tight_layout();fig.savefig(OUT/'interaction_estimates_by_model.png',dpi=170,bbox_inches='tight');plt.show()

pay_grid=np.quantile(s.income_c,[.1,.25,.5,.75,.9])
fig,axes=plt.subplots(1,3,figsize=(17,4.8),sharey=True)
for ax,c in zip(axes,crime_types):
    names=['crime_'+c+'_c','int_'+c+'_income']
    X=s[names+['income_c']+prop+loc+econ].astype(float)
    scale=X.std().replace(0,1).to_numpy();Xs=X/scale
    absorbed=pd.DataFrame({k:pd.Categorical(s[k]) for k in ['lsoacode','yearquarter_fe']},index=s.index)
    fit=AbsorbingLS(s.lnprice,Xs,absorb=absorbed,drop_absorbed=True).fit(cov_type='clustered',clusters=s.lsoacode,debiased=True)
    b=fit.params.loc[names].to_numpy()/scale[:2]
    V=fit.cov.loc[names,names].to_numpy()/np.outer(scale[:2],scale[:2])
    design=np.column_stack([np.ones(len(pay_grid)),pay_grid])
    slope=design@b;se=np.sqrt(np.einsum('ij,jk,ik->i',design,V,design))
    crit=stats.t.ppf(.975,s.lsoacode.nunique()-1)
    ax.plot(pay_grid+s.grossannualpay_k.mean(),slope,marker='o')
    ax.fill_between(pay_grid+s.grossannualpay_k.mean(),slope-crit*se,slope+crit*se,alpha=.2)
    ax.axhline(0,color='grey',linestyle='--');ax.set_title(c.replace('mps_','').title())
    ax.set_xlabel('Borough annual pay (GBP 1,000)')
axes[0].set_ylabel('M7 crime slope (95% clustered CI)')
fig.tight_layout();fig.savefig(OUT/'m7_crime_slopes_by_income.png',dpi=170,bbox_inches='tight');plt.show()


/tmp/ipykernel_1690528/1838401709.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'sample_and_income_patterns.png',dpi=170,bbox_inches='tight');plt.show()


/tmp/ipykernel_1690528/1838401709.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'interaction_estimates_by_model.png',dpi=170,bbox_inches='tight');plt.show()


/tmp/ipykernel_1690528/1838401709.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'m7_crime_slopes_by_income.png',dpi=170,bbox_inches='tight');plt.show()
